## 1. Your First Agent

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.tools import StructuredTool
from langchain.agents import initialize_agent, AgentType

llm = ChatOpenAI(
    temperature=0.1
)

def plus(a, b):
    return a + b

agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION, 
    tools=[
        StructuredTool.from_function(
            func=plus, 
            name="Sum Calculator", 
            description="Use this to perform sums of two numbers. This tool take two arguments, both  should be numbers. till finish."
        )
    ]
)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

agent.invoke(prompt)





> Entering new AgentExecutor chain...
Thought: I can use the Sum Calculator tool to find the total cost of all these amounts added together.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 355.39, "b": 924.87}
}
```
Observation: 1280.26
Thought:I have calculated the sum of the first two amounts. Now, I will continue to find the total sum by adding the remaining amounts.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 1280.26, "b": 721.2}
}
```
Observation: 2001.46
Thought:I have calculated the sum of the first three amounts. Now, I will continue to find the total sum by adding the remaining amounts.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 2001.46, "b": 1940.29}
}
```
Observation: 3941.75
Thought:I have calculated the sum of the first four amounts. Now, I will continue to find the total sum by adding the remaining amounts.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 3941.75, "b": 573.63}
}
```


{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost is $5244.26'}

## 2. Zero-shot ReAct Agent 

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType

llm = ChatOpenAI(
    temperature=0.1
)

def plus(inputs):
    a, b = inputs.split(",")
    return float(a) + float(b)

agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    tools=[
        Tool.from_function(
            func=plus,
            name="Sum Calculator",
            description="Use this to perform sums of two numbers. Use this tool by sending a pair of number separated by a comma.\nExample:1,2",
        ),
    ],
)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

agent.invoke(prompt)





> Entering new AgentExecutor chain...
I need to add up all these numbers to find the total cost.
Action: Sum Calculator
Action Input: 355.39, 924.87
Observation: 1280.26
Thought:I need to continue adding the next number to the previous sum.
Action: Sum Calculator
Action Input: 1280.26, 721.2
Observation: 2001.46
Thought:I need to continue adding the next number to the previous sum.
Action: Sum Calculator
Action Input: 2001.46, 1940.29
Observation: 3941.75
Thought:I need to continue adding the next number to the previous sum.
Action: Sum Calculator
Action Input: 3941.75, 573.63
Observation: 4515.38
Thought:I need to continue adding the next number to the previous sum.
Action: Sum Calculator
Action Input: 4515.38, 65.72
Observation: 4581.1
Thought:I need to continue adding the next number to the previous sum.
Action: Sum Calculator
Action Input: 4581.1, 35.00
Observation: 4616.1
Thought:I need to continue adding the next number to the previous sum.
Action: Sum Calculator
Action Input: 

{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost is $5273.38'}

## 3. OpenAI Functions Agent

In [1]:
from typing import Type
from langchain.chat_models import ChatOpenAI
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from langchain.agents import initialize_agent, AgentType

llm = ChatOpenAI(
    temperature=0.1
)

def plus(a, b):
    return a + b

class CalculatorToolArgsSchema(BaseModel):
    a: float = Field(description="The first number")
    b: float = Field(description="The second number")

class CalculatorTool(BaseTool):
    name = "CalculatorTool"
    description = """
    Use this to perform sums of two numbers.
    The first and second arguments should be numbers.
    Only receives two arguments.
    till finish.
    """
    args_schema: Type[CalculatorToolArgsSchema] = CalculatorToolArgsSchema

    def _run(self, a, b):
        return a + b

agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        CalculatorTool()
    ],
)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `CalculatorTool` with `{'a': 355.39, 'b': 924.87}`


1280.26
Invoking: `CalculatorTool` with `{'a': 1280.26, 'b': 721.2}`


2001.46
Invoking: `CalculatorTool` with `{'a': 2001.46, 'b': 1940.29}`


3941.75
Invoking: `CalculatorTool` with `{'a': 3941.75, 'b': 573.63}`


4515.38
Invoking: `CalculatorTool` with `{'a': 4515.38, 'b': 65.72}`


4581.1
Invoking: `CalculatorTool` with `{'a': 4581.1, 'b': 35.0}`


4616.1
Invoking: `CalculatorTool` with `{'a': 4616.1, 'b': 552.0}`


5168.1
Invoking: `CalculatorTool` with `{'a': 5168.1, 'b': 76.16}`


5244.26
Invoking: `CalculatorTool` with `{'a': 5244.26, 'b': 29.12}`


5273.38The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.

> Finished chain.


{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.'}

## 4. Search Tool

In [3]:
from typing import Type
from langchain.chat_models import ChatOpenAI
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchResults

llm = ChatOpenAI(temperature=0.1)

class StockMarketSymbolSearchToolArgsSchema(BaseModel):
    query: str = Field(
        description="The query you will search for.Example query: Stock Market Symbol for Apple Company"
    )


class StockMarketSymbolSearchTool(BaseTool):
    name = "StockMarketSymbolSearchTool"
    description = """
    Use this tool to find the stock market symbol for a company.
    It takes a query as an argument.
    Example query: Stock Market Symbol for Apple Company
    """
    args_schema: Type[
        StockMarketSymbolSearchToolArgsSchema
    ] = StockMarketSymbolSearchToolArgsSchema

    def _run(self, query):
        ddg = DuckDuckGoSearchResults()
        return ddg.run(query)


agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        StockMarketSymbolSearchTool(),
    ],
)

prompt = "Give me information on Cloudflare's stock and help me analyze if it's a potential good investment. Also tell me what symbol does the stock have."

agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `StockMarketSymbolSearchTool` with `{'query': 'Stock Market Symbol for Cloudflare'}`




/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:82: UserWarning: backend='api' is deprecated, using backend='auto'
  results = ddgs.text(


[snippet: 방문 중인 사이트에서 설명을 제공하지 않습니다., title: Naver, link: http://stock.naver.com/], [snippet: Buy and sell the hottest sneakers including Adidas Yeezy and Retro Jordans, Supreme streetwear, trading cards, collectibles, …, title: StockX: Sneakers, Streetwear, Trading Cards, Handbags, Watches, link: https://stockx.com/], [snippet: Google Finance provides real-time market quotes, international stock information, financial news, currency conversion …, title: Stock Market Prices, Real-time Quotes & Business News - Google, link: https://www.google.com/finance/], [snippet: Select a stock photo, stunning vector image, HD video clip, or Adobe Photoshop template to help tell your story — then …, title: Stock Photos, Images, Videos & AI Editing Tools | Adobe Stock, link: https://stock.adobe.com/]
Invoking: `StockMarketSymbolSearchTool` with `{'query': 'Stock Market Symbol for Cloudflare'}`
responded: I'm sorry, but I couldn't find the stock market symbol for Cloudflare. Let me try another source 

/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:82: UserWarning: backend='api' is deprecated, using backend='auto'
  results = ddgs.text(


[snippet: 4 hours ago · A detailed overview of Cloudflare, Inc. (NET) stock, including real-time price, …, title: Cloudflare (NET) Stock Price & Overview, link: https://stockanalysis.com/stocks/net/], [snippet: Cloudflare, Inc. is a connectivity cloud company. The Company delivers a range of services to businesses of all sizes and in all geographies, …, title: Cloudflare, Inc. Stock (NET) - Quote NYSE- MarketScreener, link: https://www.marketscreener.com/quote/stock/CLOUDFLARE-INC-65956823/], [snippet: Aug 9, 2026 · Explore the Cloudflare stock price history with detailed daily historical prices, including open, high, low, close, and volume …, title: Cloudflare Stock Price History - Investing.com, link: https://www.investing.com/equities/cloudflare-inc-historical-data], [snippet: 3 hours ago · Last Updated 2026-09-09 4:04 PM ET., title: Cloudflare, Inc. - Stock Quote & Chart, link: https://www.cloudflare.net/stock-information/Stock-Quote--Chart/default.aspx]
Invoking: `StockMarketSymbo

/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:82: UserWarning: backend='api' is deprecated, using backend='auto'
  results = ddgs.text(


The stock market symbol for Cloudflare is NET. Now, let's analyze Cloudflare's stock to see if it's a potential good investment. To do this analysis, we need to look at various factors such as the company's financial performance, growth prospects, industry trends, and market conditions. Let's start by looking at Cloudflare's recent stock performance and key statistics.
I will now gather information on Cloudflare's recent stock performance, key statistics, financial performance, growth prospects, industry trends, and market conditions to help analyze if it's a potential good investment. Let me gather the necessary data for the analysis.

> Finished chain.


{'input': "Give me information on Cloudflare's stock and help me analyze if it's a potential good investment. Also tell me what symbol does the stock have.",
 'output': "The stock market symbol for Cloudflare is NET. Now, let's analyze Cloudflare's stock to see if it's a potential good investment. To do this analysis, we need to look at various factors such as the company's financial performance, growth prospects, industry trends, and market conditions. Let's start by looking at Cloudflare's recent stock performance and key statistics.\nI will now gather information on Cloudflare's recent stock performance, key statistics, financial performance, growth prospects, industry trends, and market conditions to help analyze if it's a potential good investment. Let me gather the necessary data for the analysis."}